In [117]:
import pandas as pd
import geopandas as gpd

gdf = gpd.read_file('../../Data_list/preprocessing_result/군집화포인트_등고선_인구병합결과/위치_등고선_시군구_비율.shp', encoding='cp949')
gdf.head()

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535)
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258)
2,2,1435,부산광역시 금정구,0.061106,50.0,20.0,130.0,30.0,80.0,POINT (129.09771 35.21647)
3,3,1772,부산광역시 해운대구,0.084278,35.0,35.0,145.0,0.0,110.0,POINT (129.15593 35.23157)
4,4,2419,부산광역시 해운대구,0.084278,125.0,40.0,285.0,85.0,160.0,POINT (129.1311 35.19808)


In [118]:
candidate_gdf = pd.read_csv('../../Data_list/preprocessing_result/스코어링할데이터/candidate_gdf.csv')
candidate_gdf.head()

,cluster_id,data_point,geometry,residences_count,bus_count,train_count,parking_count,children_care_count,어린이,high_up,high_down
0,0,1976,POINT (129.10029965278912 35.21534903840932),13966,5,0,0,16,0.061106,25.0,90.0
1,1,1883,POINT (129.15085966138363 35.22258067813977),5294,6,0,0,7,0.084278,35.0,155.0
2,2,1435,POINT (129.09771423446185 35.21646805222987),13815,3,0,0,12,0.061106,30.0,80.0
3,3,1772,POINT (129.1559338144019 35.23156637192356),5347,5,0,1,12,0.084278,0.0,110.0
4,4,2419,POINT (129.131100786837 35.19807624497906),5430,4,0,1,12,0.084278,85.0,160.0


In [119]:
# 중복 컬럼을 제외한 candidate_gdf의 컬럼만 추출
duplicate_cols = set(gdf.columns) & set(candidate_gdf.columns)
candidate_gdf_unique = candidate_gdf[[col for col in candidate_gdf.columns if col not in duplicate_cols]]

# gdf와 candidate_gdf_unique를 컬럼 기준으로 합침 (index 맞추기)
merged_candidate_df = pd.concat([gdf.reset_index(drop=True), candidate_gdf_unique.reset_index(drop=True)], axis=1)

# 중복되는 데이터가 들어간 컬럼 제거
merged_candidate_df = merged_candidate_df.drop(columns=['어린이'])

In [120]:
# 일부 object 형태의 컬럼의 데이터타입 모두 숫자로 변경
merged_candidate_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']] = merged_candidate_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']].astype('float64')
merged_candidate_df.dtypes

cluster_id                int64
data_point                int64
시군구                      object
어린이비율                   float64
contour                 float64
contour_mi              float64
contour_ma              float64
high_up                 float64
high_down               float64
geometry               geometry
residences_count          int64
bus_count                 int64
train_count               int64
parking_count             int64
children_care_count       int64
dtype: object

In [121]:
merged_candidate_df.head(2)

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,residences_count,bus_count,train_count,parking_count,children_care_count
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535),13966,5,0,0,16
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258),5294,6,0,0,7


In [122]:
# 각 컬럼별 점수 계산 함수 정의

def score_bus_count(x):
    # 0~1: 0.5점, 2~3: 1점, 4~5: 1.5점, ... 18~19: 5점
    return 0.5 + (x // 2) * 0.5

def score_train_count(x):
    # 0: 0점, 1: 1점, 2: 2점
    return x

def score_parking_count(x):
    # 0: 0점, 1: 1점, 2: 2점, 3 이상: 3점
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

def score_child_ratio(x):
    # ~0.03: 1점, 0.03~0.06: 2점, 0.06~0.09: 3점, 0.09~0.12: 4점, 0.12~: 5점
    if x <= 0.03:
        return 1
    elif x <= 0.06:
        return 2
    elif x <= 0.09:
        return 3
    elif x <= 0.12:
        return 4
    else:
        return 5

def score_children_care_count(x):
    # 0~2: 1점, 3~5: 2점, 6~8: 3점, 9~11: 4점, 12~14: 5점, 15~17: 6점, 18~20: 7점, 21~23: 8점, 24~: 9점
    return 1 + (x // 3)

def score_high_diff(up, down):
    # 둘 다 40 이하: 4점, 둘 중 하나만 40 초과: 2점, 둘 다 40 초과: 0점
    if up <= 40 and down <= 40:
        return 4
    elif up > 40 and down > 40:
        return 0
    else:
        return 2

def score_residences_count(x):
    # 0~1999: 1점, 2000~3999: 2점, ... 18000 이상: 10점
    return 1 + (x // 2000)

# 각 점수 컬럼 생성
merged_candidate_df['score_bus'] = merged_candidate_df['bus_count'].apply(score_bus_count)
merged_candidate_df['score_train'] = merged_candidate_df['train_count'].apply(score_train_count)
merged_candidate_df['score_parking'] = merged_candidate_df['parking_count'].apply(score_parking_count)
merged_candidate_df['score_child_ratio'] = merged_candidate_df['어린이비율'].apply(score_child_ratio)
merged_candidate_df['score_children_care'] = merged_candidate_df['children_care_count'].apply(score_children_care_count)
merged_candidate_df['score_high'] = merged_candidate_df.apply(lambda row: score_high_diff(row['high_up'], row['high_down']), axis=1)
merged_candidate_df['score_residences'] = merged_candidate_df['residences_count'].apply(score_residences_count)

# 총점 컬럼 추가
score_cols = [
    'score_bus', 'score_train', 'score_parking', 'score_child_ratio',
    'score_children_care', 'score_high', 'score_residences'
]
merged_candidate_df['total_score'] = merged_candidate_df[score_cols].sum(axis=1)

merged_candidate_df.to_csv('../../Data_list/candidate_result/total_score.csv')

In [123]:
# 결과 확인
merged_candidate_df.head(2)

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535),...,0,16,1.5,0,0,3,6,2,7,19.5
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258),...,0,7,2.0,0,0,3,3,2,3,13.0


In [124]:
merged_candidate_df.describe()

,cluster_id,data_point,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,residences_count,bus_count,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
count,1140.000000,1140.000000,1139.000000,1140.000000,1123.000000,1123.000000,1123.000000,1123.000000,1140.000000,1140.000000,...,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000
mean,569.500000,154.673684,0.102829,42.342105,21.424755,128.499555,20.017809,87.056990,3613.900000,3.058772,...,0.394737,3.121053,1.197807,0.032456,0.373684,3.744737,1.849123,2.307018,2.421053,11.925877
std,329.233959,262.580034,0.035921,55.400185,37.794835,103.276737,24.261823,65.758283,4270.311409,3.222924,...,0.834068,4.156433,0.788217,0.182171,0.718995,0.902011,1.305102,1.282195,2.063499,3.098045
min,0.000000,1.000000,0.029072,0.000000,-5.000000,0.000000,0.000000,0.000000,2.000000,0.000000,...,0.000000,0.000000,0.500000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,5.500000
25%,284.750000,7.000000,0.071018,5.000000,0.000000,45.000000,0.000000,25.000000,521.000000,0.000000,...,0.000000,0.000000,0.500000,0.000000,0.000000,3.000000,1.000000,2.000000,1.000000,9.500000
50%,569.500000,36.000000,0.113239,25.000000,5.000000,120.000000,10.000000,85.000000,1079.500000,2.000000,...,0.000000,1.000000,1.000000,0.000000,0.000000,4.000000,1.000000,2.000000,1.000000,11.500000
75%,854.250000,194.000000,0.154658,60.000000,25.000000,185.000000,30.000000,135.000000,5955.500000,5.000000,...,1.000000,5.000000,1.500000,0.000000,1.000000,5.000000,2.000000,4.000000,3.000000,14.000000
max,1139.000000,2419.000000,0.154658,405.000000,315.000000,545.000000,150.000000,275.000000,16985.000000,18.000000,...,8.000000,24.000000,5.000000,2.000000,3.000000,5.000000,9.000000,4.000000,9.000000,24.000000


In [125]:
# total_score 기준으로 내림차순 정렬 후 상위 100개 추출
top100_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(100)
top300_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(300)

In [126]:
top100_df.to_file('../../Data_list/candidate_result/top100.shp', encoding='cp949')
top300_df.to_file('../../Data_list/candidate_result/top300.shp', encoding='cp949')

C:\Users\tjral\AppData\Local\Temp\ipykernel_32064\313668103.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  top100_df.to_file('../../Data_list/candidate_result/top100.shp', encoding='cp949')
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'residences_count' to 'residences'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'train_count' to 'train_coun'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parking_count' to 'parking_co'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'children_care_count' to 'children_c'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 's

## QGIS에서 확인한 결과 거주지가 많은 중심에 너무 쏠리는 문제가 발생
## 거주지 개수에 가중치를 낮춰서 다시 점수화

In [127]:
# 각 컬럼별 점수 계산 함수 정의

def score_bus_count(x):
    # 0~1: 0.5점, 2~3: 1점, 4~5: 1.5점, ... 18~19: 5점
    return 0.5 + (x // 2) * 0.5

def score_train_count(x):
    # 0: 0점, 1: 1점, 2: 2점
    return x

def score_parking_count(x):
    # 0: 0점, 1: 1점, 2: 2점, 3 이상: 3점
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

def score_child_ratio(x):
    # ~0.03: 1점, 0.03~0.06: 2점, 0.06~0.09: 3점, 0.09~0.12: 4점, 0.12~: 5점
    if x <= 0.03:
        return 1
    elif x <= 0.06:
        return 2
    elif x <= 0.09:
        return 3
    elif x <= 0.12:
        return 4
    else:
        return 5

def score_children_care_count(x):
    # 0~2: 1점, 3~5: 2점, 6~8: 3점, 9~11: 4점, 12~14: 5점, 15~17: 6점, 18~20: 7점, 21~23: 8점, 24~: 9점
    return 1 + (x // 3)

def score_high_diff(up, down):
    # 둘 다 40 이하: 4점, 둘 중 하나만 40 초과: 2점, 둘 다 40 초과: 0점
    if up <= 40 and down <= 40:
        return 4
    elif up > 40 and down > 40:
        return 0
    else:
        return 2

def score_residences_count(x):
    # 0~1999: 1점, 2000~3999: 2점, ... 18000 이상: 10점
    return (1 + (x // 2000)) * 0.5 # 가중치를 반으로 낮춤

# 각 점수 컬럼 생성
merged_candidate_df['score_bus'] = merged_candidate_df['bus_count'].apply(score_bus_count)
merged_candidate_df['score_train'] = merged_candidate_df['train_count'].apply(score_train_count)
merged_candidate_df['score_parking'] = merged_candidate_df['parking_count'].apply(score_parking_count)
merged_candidate_df['score_child_ratio'] = merged_candidate_df['어린이비율'].apply(score_child_ratio)
merged_candidate_df['score_children_care'] = merged_candidate_df['children_care_count'].apply(score_children_care_count)
merged_candidate_df['score_high'] = merged_candidate_df.apply(lambda row: score_high_diff(row['high_up'], row['high_down']), axis=1)
merged_candidate_df['score_residences'] = merged_candidate_df['residences_count'].apply(score_residences_count)

# 총점 컬럼 추가
score_cols = [
    'score_bus', 'score_train', 'score_parking', 'score_child_ratio',
    'score_children_care', 'score_high', 'score_residences'
]
merged_candidate_df['total_score'] = merged_candidate_df[score_cols].sum(axis=1)

merged_candidate_df.to_csv('../../Data_list/candidate_result/total_score_0.5.csv')

In [128]:
# total_score 기준으로 내림차순 정렬 후 상위 100개 추출
top100_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(100)
top300_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(300)

In [129]:
top100_df.to_file('../../Data_list/candidate_result/top100_0.5.shp', encoding='cp949')
top300_df.to_file('../../Data_list/candidate_result/top300_0.5.shp', encoding='cp949')

C:\Users\tjral\AppData\Local\Temp\ipykernel_32064\2148677008.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  top100_df.to_file('../../Data_list/candidate_result/top100_0.5.shp', encoding='cp949')
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'residences_count' to 'residences'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'train_count' to 'train_coun'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parking_count' to 'parking_co'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'children_care_count' to 'children_c'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field nam

## QGIS에서 확인한 결과 이전 데이터보다 퍼져 있지만 그래도 중심에 쏠리는 문제가 해결되지 않음
## 점수 합산에 거주지 개수를 제거하고 다시 점수화

In [130]:
# 각 컬럼별 점수 계산 함수 정의

def score_bus_count(x):
    # 0~1: 0.5점, 2~3: 1점, 4~5: 1.5점, ... 18~19: 5점
    return 0.5 + (x // 2) * 0.5

def score_train_count(x):
    # 0: 0점, 1: 1점, 2: 2점
    return x

def score_parking_count(x):
    # 0: 0점, 1: 1점, 2: 2점, 3 이상: 3점
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

def score_child_ratio(x):
    # ~0.03: 1점, 0.03~0.06: 2점, 0.06~0.09: 3점, 0.09~0.12: 4점, 0.12~: 5점
    if x <= 0.03:
        return 1
    elif x <= 0.06:
        return 2
    elif x <= 0.09:
        return 3
    elif x <= 0.12:
        return 4
    else:
        return 5

def score_children_care_count(x):
    # 0~2: 1점, 3~5: 2점, 6~8: 3점, 9~11: 4점, 12~14: 5점, 15~17: 6점, 18~20: 7점, 21~23: 8점, 24~: 9점
    return 1 + (x // 3)

def score_high_diff(up, down):
    # 둘 다 40 이하: 4점, 둘 중 하나만 40 초과: 2점, 둘 다 40 초과: 0점
    if up <= 40 and down <= 40:
        return 4
    elif up > 40 and down > 40:
        return 0
    else:
        return 2

# 거주지는 점수 부여하지 않으므로 주석 처리
# def score_residences_count(x):
#     # 0~1999: 1점, 2000~3999: 2점, ... 18000 이상: 10점
#     return (1 + (x // 2000)) * 0.5 # 가중치를 반으로 낮춤

# 각 점수 컬럼 생성
merged_candidate_df['score_bus'] = merged_candidate_df['bus_count'].apply(score_bus_count)
merged_candidate_df['score_train'] = merged_candidate_df['train_count'].apply(score_train_count)
merged_candidate_df['score_parking'] = merged_candidate_df['parking_count'].apply(score_parking_count)
merged_candidate_df['score_child_ratio'] = merged_candidate_df['어린이비율'].apply(score_child_ratio)
merged_candidate_df['score_children_care'] = merged_candidate_df['children_care_count'].apply(score_children_care_count)
merged_candidate_df['score_high'] = merged_candidate_df.apply(lambda row: score_high_diff(row['high_up'], row['high_down']), axis=1)
# 거주지는 점수 부여하지 않으므로 주석 처리
# merged_candidate_df['score_residences'] = merged_candidate_df['residences_count'].apply(score_residences_count)

# 총점 컬럼 추가
score_cols = [
    'score_bus', 'score_train', 'score_parking', 'score_child_ratio',
    'score_children_care', 'score_high' # 'score_residence' 삭제
]
merged_candidate_df['total_score'] = merged_candidate_df[score_cols].sum(axis=1)

merged_candidate_df.to_csv('../../Data_list/candidate_result/total_score_no.csv')

In [131]:
# total_score 기준으로 내림차순 정렬 후 상위 100개 추출
top100_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(100)
top300_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(300)

In [132]:
top100_df.to_file('../../Data_list/candidate_result/top100_no.shp', encoding='cp949')
top300_df.to_file('../../Data_list/candidate_result/top300_no.shp', encoding='cp949')

C:\Users\tjral\AppData\Local\Temp\ipykernel_32064\1688905484.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  top100_df.to_file('../../Data_list/candidate_result/top100_no.shp', encoding='cp949')
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'residences_count' to 'residences'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'train_count' to 'train_coun'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parking_count' to 'parking_co'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'children_care_count' to 'children_c'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name